# **1. Dataset Acquisition and Setup**

1.1 Download and Extract Data

In [1]:
# Download dataset from UCI Repository
!wget https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip

--2026-08-29 23:07:32--  https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘individual+household+electric+power+consumption.zip’

individual+househol     [     <=>            ]  19.68M  23.2MB/s    in 0.8s    

2026-08-29 23:07:34 (23.2 MB/s) - ‘individual+household+electric+power+consumption.zip’ saved [20640916]



In [2]:
# Unzip the downloaded archive
!unzip /content/individual+household+electric+power+consumption.zip

Archive:  /content/individual+household+electric+power+consumption.zip
  inflating: household_power_consumption.txt  


In [3]:
# Inspect the first few lines of the raw text file
! head /content/household_power_consumption.txt

Date;Time;Global_active_power;Global_reactive_power;Voltage;Global_intensity;Sub_metering_1;Sub_metering_2;Sub_metering_3
16/12/2006;17:24:00;4.216;0.418;234.840;18.400;0.000;1.000;17.000
16/12/2006;17:25:00;5.360;0.436;233.630;23.000;0.000;1.000;16.000
16/12/2006;17:26:00;5.374;0.498;233.290;23.000;0.000;2.000;17.000
16/12/2006;17:27:00;5.388;0.502;233.740;23.000;0.000;1.000;17.000
16/12/2006;17:28:00;3.666;0.528;235.680;15.800;0.000;1.000;17.000
16/12/2006;17:29:00;3.520;0.522;235.020;15.000;0.000;2.000;17.000
16/12/2006;17:30:00;3.702;0.520;235.090;15.800;0.000;1.000;17.000
16/12/2006;17:31:00;3.700;0.520;235.220;15.800;0.000;1.000;17.000
16/12/2006;17:32:00;3.668;0.510;233.990;15.800;0.000;1.000;17.000


In [4]:
import sqlite3
import pandas as pd
import csv

In [5]:
# Path to the unzipped raw dataset
raw_data_path='/content/household_power_consumption.txt'

# **2. Data Loading & Pandas-SQLite Integration**

2.1 Load Raw Data into Pandas DataFrame

In [6]:
df = pd.read_csv(raw_data_path, sep=';', low_memory=False)
print(df.shape[1], df.shape[0])

9 2075259


In [7]:
df.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0


2.2 Export DataFrame to SQLite Database (pandas approach)

In [9]:
#create SQLite database
conn1 = sqlite3.connect('household_power.db')

df.to_sql('household_power', conn1, if_exists='replace', index=False, dtype={
    'Date': 'TEXT',
    'Time': 'TEXT',
    'Global_active_power': 'REAL',
    'Global_reactive_power': 'REAL',
    'Voltage': 'REAL',
    'Global_intensity': 'REAL',
    'Sub_metering_1': 'REAL',
    'Sub_metering_2': 'REAL',
    'Sub_metering_3': 'REAL'
})

2075259

# **3. Database Inspection & Analysis via Pandas**

3.1 Inspect Table Schema & Data Dimensions

In [10]:
# Query table column metadata
query = "PRAGMA table_info(household_power)"
result = pd.read_sql(query,conn1)
print(result)

   cid                   name  type  notnull dflt_value  pk
0    0                   Date  TEXT        0       None   0
1    1                   Time  TEXT        0       None   0
2    2    Global_active_power  REAL        0       None   0
3    3  Global_reactive_power  REAL        0       None   0
4    4                Voltage  REAL        0       None   0
5    5       Global_intensity  REAL        0       None   0
6    6         Sub_metering_1  REAL        0       None   0
7    7         Sub_metering_2  REAL        0       None   0
8    8         Sub_metering_3  REAL        0       None   0


In [11]:
# Query row and column counts
query = """
SELECT 'rows' AS dimension, COUNT(*) AS count FROM household_power
UNION ALL
SELECT 'columns' AS dimension, COUNT(*) AS count FROM pragma_table_info('household_power');
"""
result = pd.read_sql(query,conn1)
print(result)

  dimension    count
0      rows  2075259
1   columns        9


3.2 Compute Metrics & Check Null Values

In [12]:
# Compute average global active power
query = "SELECT AVG(global_active_power) AS average_power FROM household_power;"
result = pd.read_sql(query,conn1)
print(result)

   average_power
0        1.07795


In [13]:
# Count NULL values for every column in the table
query = """
SELECT
    SUM(CASE WHEN Date IS NULL THEN 1 ELSE 0 END) AS null_date,
    SUM(CASE WHEN Time IS NULL THEN 1 ELSE 0 END) AS null_time,
    SUM(CASE WHEN global_active_power IS NULL THEN 1 ELSE 0 END) AS null_global_active_power,
    SUM(CASE WHEN global_reactive_power IS NULL THEN 1 ELSE 0 END) AS null_global_reactive_power,
    SUM(CASE WHEN Voltage IS NULL THEN 1 ELSE 0 END) AS null_voltage,
    SUM(CASE WHEN global_intensity IS NULL THEN 1 ELSE 0 END) AS null_global_intensity,
    SUM(CASE WHEN sub_metering_1 IS NULL THEN 1 ELSE 0 END) AS null_sub_metering_1,
    SUM(CASE WHEN sub_metering_2 IS NULL THEN 1 ELSE 0 END) AS null_sub_metering_2,
    SUM(CASE WHEN sub_metering_3 IS NULL THEN 1 ELSE 0 END) AS null_sub_metering_3
FROM household_power;
"""
result = pd.read_sql(query,conn1)
print(result)

   null_date  null_time  null_global_active_power  null_global_reactive_power  \
0          0          0                         0                           0   

   null_voltage  null_global_intensity  null_sub_metering_1  \
0             0                      0                    0   

   null_sub_metering_2  null_sub_metering_3  
0                    0                25979  


# **4. Native SQLite & Batch Insertion (csv Module approach)**

4.1 Schema Creation using SQL DDL

In [14]:
# Connect to the second SQLite database
conn2 = sqlite3.connect('household_power2.db')

In [15]:
cursor = conn2.cursor()

In [16]:
# Create table schema if it does not exist
cursor.execute("""
CREATE TABLE IF NOT EXISTS household_power (
    Date TEXT,
    Time TEXT,
    global_active_power REAL,
    global_reactive_power REAL,
    Voltage REAL,
    global_intensity REAL,
    sub_metering_1 REAL,
    sub_metering_2 REAL,
    sub_metering_3 REAL
);
""")

conn2.commit()

4.2 Batch Insertion with executemany

In [17]:
# Open raw text file and read line by line
with open(raw_data_path,'r') as file:
  reader = csv.reader(file,delimiter=';')

  #skip header row
  header = next(reader)

  # Convert missing indicator '?' to None (SQLite NULL)
  data = [
      [None if value == '?' else value for value in row]
      for row in reader
  ]

# Prepare parameterized SQL insert query
insert_query = """
INSERT INTO household_power (
    Date, Time, global_active_power, global_reactive_power,
    Voltage, global_intensity, sub_metering_1, sub_metering_2, sub_metering_3
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);
"""
# Execute batch insertion and commit changes
cursor.executemany(insert_query,data)
#commit changes
conn2.commit()

# **5. Advanced SQL Queries & Cleanup**

5.1 Data Filtering & Aggregations

In [18]:
#query database to retrieve column names
query = "PRAGMA table_info(household_power);"
cursor.execute(query)
result = cursor.fetchall()

for column in result:
  print(column)

(0, 'Date', 'TEXT', 0, None, 0)
(1, 'Time', 'TEXT', 0, None, 0)
(2, 'global_active_power', 'REAL', 0, None, 0)
(3, 'global_reactive_power', 'REAL', 0, None, 0)
(4, 'Voltage', 'REAL', 0, None, 0)
(5, 'global_intensity', 'REAL', 0, None, 0)
(6, 'sub_metering_1', 'REAL', 0, None, 0)
(7, 'sub_metering_2', 'REAL', 0, None, 0)
(8, 'sub_metering_3', 'REAL', 0, None, 0)


In [19]:
# Count records exceeding global active power threshold (> 5.0 kW)
query = """
SELECT COUNT(*) AS count_above_threshold
FROM household_power
WHERE global_active_power > 5.0;
"""

cursor.execute(query)
result = cursor.fetchone()
print(result)

(17547,)


In [20]:
# Top 5 times of day with highest overall power consumption aggregate
query = """
SELECT Time, SUM(global_active_power) AS total_power
FROM household_power
GROUP BY Time
ORDER BY total_power DESC
LIMIT 5;
"""

cursor.execute(query)
results = cursor.fetchall()

for row in results:
  print(row)

('20:50:00', 2857.1820000000034)
('20:49:00', 2856.776)
('20:51:00', 2855.3019999999974)
('20:52:00', 2844.2799999999984)
('20:53:00', 2838.398000000001)


5.2 Close Database Connections

In [21]:
# Safely close connection to database 2
conn2.close()